# 08 — Cerrar el ciclo de RAG: generación con un LLM

**Módulo V — Diplomado de Ciencia de Datos (FES Acatlán, UNAM)**
**Sesión 11 · Bloque 1 de 2 · Duración estimada: 85 minutos**

**Prerrequisito:** `03 Aplicaciones NLP (word embeddings)/05 Embeddings/06_Using_Embeddings_in_RAG_Inference.ipynb`.
**Notas de clase:** capítulo 2, secciones *"De embeddings a recuperación de información"* y
*"Introducción a los LLMs y aplicaciones basadas en LLMs"*.

## Dónde nos quedamos

En el notebook 06 construimos los pasos 1 y 2 de RAG —indexar un corpus y recuperar los
fragmentos relevantes— y terminamos **armando el texto exacto** que se le entregaría a un modelo
de lenguaje. Nunca llamamos al modelo. Hoy cerramos el ciclo:

| Paso | Estado |
|---|---|
| 1. Indexar el corpus | Hecho en el notebook 06 |
| 2. Recuperar por similitud | Hecho en el notebook 06 |
| **3. Generar la respuesta con un LLM** | **Hoy** |

## Al terminar vas a poder

1. Llamar a un modelo de lenguaje desde Python y entender los tres roles de un mensaje
   (`system`, `user`, `assistant`).
2. **Escribir un prompt de sistema y medir su efecto**, en vez de suponerlo: vas a ver al mismo
   modelo inventar y luego negarse a inventar, cambiando únicamente esas instrucciones.
3. Comparar la respuesta del modelo con y sin contexto recuperado.
4. Elegir entre dos recuperadores con datos propios y con la restricción real del servidor donde
   se va a desplegar.

## Sobre el modelo que vamos a usar

Usamos **Groq** como proveedor y **Llama 3.3 70B** como modelo, por tres razones prácticas:
tiene una capa gratuita suficiente para una clase, es de los servicios más rápidos que existen
(cientos de tokens por segundo), y Llama es un modelo abierto, coherente con la decisión que ya
tomamos en el bloque de embeddings de usar modelos que cualquiera pueda reproducir.

Nada de lo que veremos depende de ese proveedor: el código sería casi idéntico con OpenAI,
Anthropic o Google, porque todos exponen la misma idea de una lista de mensajes con roles.

## 0. Preparación

Necesitas una llave de Groq, **gratuita**:

1. Entra a [console.groq.com/keys](https://console.groq.com/keys) y crea una cuenta.
2. Genera una llave nueva (empieza con `gsk_...`).
3. **Cópiala en ese momento**: no se vuelve a mostrar.

La celda de abajo la pide con `getpass`, que oculta lo que escribes. Nunca escribas una llave
directamente en una celda: si compartes el notebook, la compartes con ella.

In [ ]:
# --- Dependencias (instala solo lo que falte; necesario en Colab) ---
import importlib.util, subprocess, sys

EN_COLAB = "google.colab" in sys.modules

def asegurar(paquete, modulo=None):
    modulo = modulo or paquete.replace("-", "_")
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

for paquete, modulo in [("groq", "groq"), ("scikit-learn", "sklearn"),
                        ("pandas", "pandas"), ("numpy", "numpy")]:
    asegurar(paquete, modulo)

print("Dependencias listas." + ("   (Google Colab detectado)" if EN_COLAB else ""))

In [ ]:
import getpass
import os

import numpy as np
import pandas as pd
from groq import Groq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Pega aquí tu llave de Groq: ")

cliente = Groq(api_key=os.environ["GROQ_API_KEY"])
MODELO = "llama-3.3-70b-versatile"

print("Cliente listo.")

## 1. La primera llamada: los tres roles de un mensaje

Un modelo de chat no recibe "un texto": recibe una **lista de mensajes**, cada uno con un rol.

| Rol | Quién habla | Para qué sirve |
|---|---|---|
| `system` | El desarrollador (tú) | Fija el comportamiento antes de que el usuario diga nada: papel, idioma, tono, límites. |
| `user` | La persona | La pregunta o instrucción concreta. |
| `assistant` | El modelo | Sus respuestas anteriores. Es lo que le da memoria de la conversación. |

El modelo es **sin estado**: no recuerda nada entre llamadas. Lo que llamamos "memoria" es que
nosotros le reenviamos la lista completa cada vez.

In [ ]:
def preguntar_al_modelo(mensajes, temperatura=0.2, max_tokens=700, modelo=MODELO):
    """Envía una lista de mensajes al modelo y devuelve el texto de la respuesta."""
    respuesta = cliente.chat.completions.create(
        model=modelo,
        messages=mensajes,
        temperature=temperatura,
        max_tokens=max_tokens,
    )
    return respuesta.choices[0].message.content


mensajes = [
    {"role": "system", "content": "Eres un profesor de estadística. Responde en español, en dos oraciones."},
    {"role": "user",   "content": "¿Qué es la esperanza condicional?"},
]

print(preguntar_al_modelo(mensajes))

In [ ]:
# El mismo mensaje del usuario, con otro prompt de sistema.
# Cambia SOLO la primera línea y observa cuánto cambia la respuesta.
for rol in [
    "Eres un profesor de estadística. Responde en español, en dos oraciones.",
    "Eres un divulgador que explica con analogías cotidianas, sin fórmulas. Responde en dos oraciones.",
    "Eres un revisor de tesis exigente. Señala qué le falta a la pregunta para ser precisa. Dos oraciones.",
]:
    print(f"--- SYSTEM: {rol[:60]}...")
    print(preguntar_al_modelo([
        {"role": "system", "content": rol},
        {"role": "user",   "content": "¿Qué es la esperanza condicional?"},
    ]))
    print()

El prompt de sistema no es decorativo: **es la parte del programa que define el
comportamiento del componente**. La misma pregunta produce tres respuestas distintas en registro,
formato y nivel de detalle. En el resto del notebook lo vamos a usar para algo mucho más serio que
el tono: para evitar que el modelo invente.

## 2. El problema que RAG viene a resolver

Antes de construir nada, veamos el problema en vivo. Le vamos a preguntar al modelo algo que
**no puede saber**: información específica de este diplomado, que no estaba en sus datos de
entrenamiento.

In [ ]:
pregunta_interna = ("¿Cómo se reparte la calificación del Módulo V del Diplomado de Ciencia de "
                    "Datos de la FES Acatlán, y qué peso tiene cada tarea?")

respuesta_sin_contexto = preguntar_al_modelo([
    {"role": "system", "content": "Eres un asistente académico. Responde en español."},
    {"role": "user",   "content": pregunta_interna},
])

print(respuesta_sin_contexto)

**Lee la respuesta con cuidado y clasifícala.** Vas a encontrarte con una de estas tres:

1. El modelo dice honestamente que no tiene esa información. Es el mejor caso, y no siempre pasa.
2. El modelo **inventa** una distribución de calificaciones que suena perfectamente razonable
   —porcentajes redondos, criterios plausibles— y **no avisa que la está inventando**. Esto es una
   alucinación, y es peligrosa precisamente porque es verosímil.
3. El modelo responde en términos genéricos sobre "los diplomados en general", esquivando la
   pregunta concreta.

Ninguna de las tres sirve. El modelo no tiene el dato, y no hay prompt que lo haga aparecer. La
única solución es **dárselo**.

## 3. La base de conocimiento

Nuestro corpus son 29 fragmentos que resumen los temas del módulo, uno por idea. Cada fragmento
trae su número de sesión, que después usaremos para citar la fuente.

> Este es literalmente el mismo archivo (`corpus_diplomado.py`) que usa la aplicación que vamos a
> desplegar en la segunda mitad de la sesión. Lo dejamos aquí completo para que puedas leerlo y,
> sobre todo, para que puedas **sustituirlo por contenido tuyo**.

In [ ]:
CORPUS = [
    {"sesion": 1, "titulo": "Esperanza condicional",
     "texto": "La esperanza condicional E[Y|X] es el valor promedio de Y dado que conocemos X. "
              "Es el objeto que en el fondo estiman casi todos los modelos del módulo: la regresión "
              "lineal la aproxima con una función lineal de X, y los modelos de respuesta binaria "
              "con una función no lineal acotada entre cero y uno."},

    {"sesion": 1, "titulo": "DAG, confusores y colisionadores",
     "texto": "Un grafo acíclico dirigido, o DAG, representa los supuestos causales de un problema "
              "mediante flechas entre variables. Sirve para decidir por qué variables hay que "
              "controlar. Un confusor abre una ruta trasera que sesga el efecto estimado y hay que "
              "controlarlo; en cambio, controlar por un colisionador introduce sesgo de selección "
              "donde antes no había, así que hay que dejarlo fuera."},

    {"sesion": 2, "titulo": "Mínimos cuadrados ordinarios",
     "texto": "El estimador de mínimos cuadrados ordinarios, MCO, elige los coeficientes que "
              "minimizan la suma de los residuos al cuadrado. Bajo los supuestos clásicos es "
              "insesgado y tiene la varianza mínima entre los estimadores lineales insesgados."},

    {"sesion": 2, "titulo": "Bondad de ajuste y R cuadrada",
     "texto": "El coeficiente de determinación R cuadrada mide la proporción de la varianza de la "
              "variable dependiente que el modelo explica. Nunca baja al agregar variables, así que "
              "dentro de muestra no sirve para comparar modelos con distinto número de regresores: "
              "para eso hay que mirar el desempeño fuera de muestra."},

    {"sesion": 2, "titulo": "Sobreajuste y separación de la muestra",
     "texto": "Separar los datos en un conjunto de entrenamiento y uno de prueba permite estimar el "
              "error fuera de muestra. Un modelo puede memorizar el ruido del entrenamiento y lucir "
              "excelente ahí, pero fallar con datos nuevos: eso es sobreajuste, y solo se detecta "
              "evaluando en datos que el modelo no vio al ajustarse."},

    {"sesion": 2, "titulo": "Regresión Ridge",
     "texto": "La regresión Ridge agrega a la función objetivo una penalización proporcional a la "
              "suma de los coeficientes al cuadrado. Encoge los coeficientes hacia cero y reduce la "
              "varianza del estimador cuando hay colinealidad entre los regresores, pero nunca los "
              "deja exactamente en cero, así que conserva todas las variables."},

    {"sesion": 2, "titulo": "Regresión Lasso y selección de variables",
     "texto": "La regresión Lasso agrega una penalización proporcional a la suma de los valores "
              "absolutos de los coeficientes. A diferencia de Ridge, sí puede dejar coeficientes "
              "exactamente en cero, por lo que descarta variables y hace selección automática de "
              "predictores. Es el método a usar cuando se quiere un modelo más simple e "
              "interpretable a partir de muchas variables candidatas."},

    {"sesion": 3, "titulo": "Análisis de componentes principales",
     "texto": "El análisis de componentes principales, PCA, encuentra ejes ortogonales nuevos que "
              "capturan la mayor varianza posible de los datos. Se usa para reducir el número de "
              "dimensiones y para poder graficar en un plano de dos ejes conjuntos de datos con "
              "muchas columnas. Es una técnica no supervisada: no usa la variable a predecir."},

    {"sesion": 3, "titulo": "K-medias",
     "texto": "El algoritmo de K-medias parte las observaciones en k grupos minimizando la distancia "
              "entre cada observación y el centroide del grupo al que fue asignada. Es un método no "
              "supervisado: agrupa clientes, municipios u observaciones parecidas sin necesidad de "
              "ninguna etiqueta previa."},

    {"sesion": 4, "titulo": "Coeficiente de silueta y número de grupos",
     "texto": "El coeficiente de silueta compara qué tan cerca está una observación de su propio "
              "grupo frente al grupo vecino más cercano. Es el criterio que se usa para elegir "
              "cuántos grupos conviene formar: se prueban varios valores de k y se toma el que da "
              "mejor silueta. Se degrada en espacios de muchas dimensiones, así que sirve más para "
              "comparar alternativas que como calificación absoluta."},

    {"sesion": 4, "titulo": "Agrupamiento jerárquico",
     "texto": "El agrupamiento jerárquico construye un árbol de grupos, ya sea uniendo las "
              "observaciones más parecidas o dividiendo el conjunto completo. A diferencia de "
              "K-medias no exige fijar de antemano el número de grupos, que se decide después "
              "cortando el árbol a cierta altura."},

    {"sesion": 4, "titulo": "Deformación dinámica del tiempo (DTW)",
     "texto": "La deformación dinámica del tiempo, o DTW, compara dos series de tiempo permitiendo "
              "estirarlas o desplazarlas temporalmente, algo que la distancia euclidiana no puede "
              "hacer: dos series con la misma forma pero desfasadas se ven muy distintas para la "
              "distancia euclidiana y muy parecidas para DTW. Su costo crece con el cuadrado de la "
              "longitud de la serie, por lo que conviene reducir la frecuencia de los datos."},

    {"sesion": 5, "titulo": "Modelo logit",
     "texto": "El modelo logit supone que la probabilidad de un resultado binario sigue una función "
              "logística de un índice lineal de las covariables. Se estima por máxima verosimilitud. "
              "Sus coeficientes no se interpretan directamente como efectos marginales: hay que "
              "calcular el efecto marginal o razones de momios."},

    {"sesion": 6, "titulo": "Logit ordinal",
     "texto": "El logit ordinal se usa cuando la variable dependiente tiene categorías con un orden "
              "natural: una calificación del uno al cinco, un nivel de satisfacción, una escala de "
              "acuerdo o desacuerdo. Supone odds proporcionales, es decir, que el efecto de cada "
              "variable es el mismo entre cualquier par de categorías adyacentes."},

    {"sesion": 6, "titulo": "Matriz de confusión",
     "texto": "La matriz de confusión cruza las clases predichas contra las clases verdaderas. De "
              "ella se calculan la precisión, la sensibilidad y la exactitud, y permite ver si el "
              "modelo se equivoca más en una clase que en otra, algo que una sola cifra de exactitud "
              "esconde cuando las clases están desbalanceadas."},

    {"sesion": 7, "titulo": "Expresiones regulares y tokenización",
     "texto": "Las expresiones regulares son patrones para buscar y extraer texto. Son el primer "
              "paso del análisis de texto: sirven para limpiar, extraer campos y separar el texto en "
              "unidades (tokenizar) antes de aplicar cualquier modelo."},

    {"sesion": 7, "titulo": "Modelos de lenguaje de n-gramas",
     "texto": "Un modelo de lenguaje de n-gramas estima la probabilidad de una palabra dadas las n "
              "menos una palabras anteriores. Su limitación de fondo es que solo ve una ventana "
              "corta: no puede relacionar palabras separadas por mucha distancia dentro del texto, "
              "que es justo el problema que resolvió la arquitectura Transformer."},

    {"sesion": 8, "titulo": "Clasificador de Bayes ingenuo",
     "texto": "El clasificador de Bayes ingenuo aplica el teorema de Bayes suponiendo que los "
              "atributos son condicionalmente independientes dada la clase. Ese supuesto casi nunca "
              "se cumple en texto real, y aun así el método funciona sorprendentemente bien para "
              "clasificar documentos y detectar spam."},

    {"sesion": 9, "titulo": "Árboles de decisión y bosques aleatorios",
     "texto": "Un bosque aleatorio ajusta muchos árboles de decisión sobre muestras bootstrap y "
              "promedia sus predicciones. En cada partición considera solo un subconjunto aleatorio "
              "de predictores, y ese muestreo de variables decorrelaciona los árboles entre sí, que "
              "es lo que hace bajar la varianza frente a un solo árbol."},

    {"sesion": 9, "titulo": "Redes neuronales",
     "texto": "Una red neuronal encadena capas de transformaciones lineales seguidas de funciones de "
              "activación no lineales, como ReLU. La capa de salida usa softmax cuando el problema "
              "es de clasificación multiclase. Las capas ocultas aprenden representaciones "
              "intermedias de la entrada que sirven como embeddings."},

    {"sesion": 10, "titulo": "Qué es un word embedding",
     "texto": "Un word embedding convierte un texto en un vector denso de números reales, de modo "
              "que textos con significado parecido queden cerca entre sí. A diferencia de la bolsa "
              "de palabras, reconoce que dos palabras distintas pueden significar lo mismo, aunque "
              "no comparta ninguna letra con la otra."},

    {"sesion": 10, "titulo": "Similitud coseno",
     "texto": "La similitud coseno mide la diferencia de dirección entre dos vectores, ignorando su "
              "magnitud. Es la métrica más usada para comparar embeddings de oraciones. Cuando los "
              "vectores están normalizados coincide exactamente con el producto punto, y por eso las "
              "bases de datos vectoriales normalizan al guardar."},

    {"sesion": 10, "titulo": "Embeddings de token frente a embeddings de oración",
     "texto": "BERT produce un vector por cada token y ese vector depende del contexto, por lo que "
              "una misma palabra escrita igual, como banco, recibe representaciones distintas según "
              "el sentido en que se use. Los modelos tipo SBERT producen en cambio un solo vector "
              "por oración completa, entrenado para que la distancia entre dos oraciones refleje qué "
              "tan parecidas son en significado."},

    {"sesion": 10, "titulo": "Arquitectura Transformer y atención",
     "texto": "La arquitectura Transformer calcula pesos de atención entre todos los pares de tokens "
              "de la secuencia, lo que le permite relacionar palabras distantes entre sí. Es la base "
              "de los modelos de lenguaje grandes actuales y lo que superó la limitación de ventana "
              "corta de los modelos de n-gramas."},

    {"sesion": 10, "titulo": "Generación aumentada con recuperación (RAG)",
     "texto": "La generación aumentada con recuperación, o RAG, recupera de una base propia los "
              "documentos relevantes para una pregunta y se los entrega al modelo de lenguaje como "
              "contexto, en lugar de esperar que el modelo lo recuerde todo de su entrenamiento. "
              "Sirve para que el modelo no invente datos, para citar la fuente de cada afirmación y "
              "para usar información privada o más reciente que el modelo."},

    {"sesion": 10, "titulo": "Prompt de sistema",
     "texto": "El prompt de sistema son las instrucciones que fijan el comportamiento del modelo "
              "antes de que el usuario escriba nada: qué papel tomar, en qué idioma responder, qué "
              "tono usar y qué límites respetar. En un sistema RAG es donde se le ordena responder "
              "únicamente con el contexto recuperado, citar el fragmento y admitir cuando no tiene "
              "la información."},

    {"sesion": 10, "titulo": "Agentes y uso de herramientas",
     "texto": "Un agente es un modelo de lenguaje al que se le dan herramientas y que decide por sí "
              "mismo cuándo usarlas, siguiendo un ciclo de pensar, actuar y observar. Darle "
              "herramientas amplía lo que puede hacer, pero agrega latencia, dependencia de "
              "servicios externos, variabilidad en las respuestas y riesgos de seguridad."},

    {"sesion": 10, "titulo": "Parámetros de un modelo de lenguaje",
     "texto": "La temperatura controla qué tan variadas son las respuestas: con temperatura cero el "
              "modelo es casi determinista y con valores altos se vuelve más creativo y menos "
              "predecible. La ventana de contexto es el número máximo de tokens que el modelo puede "
              "leer de una vez, y es la razón por la que en RAG se recuperan solo unos pocos "
              "fragmentos en lugar de mandar el corpus completo."},

    {"sesion": 0, "titulo": "Evaluación del módulo",
     "texto": "La evaluación del Módulo V se reparte en partes iguales entre la Tarea 01, de "
              "regresión lineal y agrupamiento, y la Tarea 02, de análisis de texto. Cada una vale "
              "el cincuenta por ciento de la calificación del módulo. El Reporte Final pertenece al "
              "Módulo VI y se evalúa por separado."},
]

print(f"{len(CORPUS)} fragmentos")
print(f"Longitud media: {sum(len(d['texto'].split()) for d in CORPUS) / len(CORPUS):.0f} palabras")
CORPUS[6]

**Por qué los fragmentos son de este tamaño.** Un fragmento es la unidad que se recupera y
que se le entrega al modelo. Si fuera muy largo y hablara de cinco temas, su representación
quedaría "en medio" de todos y no se parecería a ninguna pregunta concreta. Si fuera demasiado
corto, la respuesta quedaría partida entre dos fragmentos y el modelo recibiría solo la mitad. La
regla práctica: **un fragmento, una idea**, entre 40 y 80 palabras.

## 4. Paso 2: recuperar (esta vez con TF-IDF)

En el notebook 06 recuperamos con embeddings. Aquí vamos a usar **TF-IDF**, la representación por
frecuencia de términos que ya conocen desde la Sesión 8. En la sección 7 comparamos las dos con
datos y explicamos por qué la app desplegada usa ésta.

Dos detalles del código que sí cambian los resultados:

- **Quitamos las palabras vacías del español.** Sin eso, preguntas como *"¿para qué sirve...?"* se
  parecen a cualquier documento solo por las palabras de relleno.
- **Repetimos el título dos veces** en el texto indexado, para que pese más que el cuerpo.

Los dos son ajustes pequeños de una línea, y más adelante veremos cuánto cambian el desempeño.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

STOPWORDS_ES = [
    "a", "al", "algo", "ante", "antes", "como", "con", "contra", "cual", "cuales", "cuando",
    "cuanto", "de", "del", "desde", "donde", "dos", "el", "ella", "ellas", "ellos", "en",
    "entre", "era", "es", "esa", "ese", "eso", "esta", "estas", "este", "esto", "estos", "ha",
    "hace", "hacer", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "más", "me",
    "mi", "mis", "mucho", "muy", "ni", "no", "nos", "o", "otra", "otro", "para", "pero", "por",
    "porque", "que", "qué", "quien", "se", "si", "sin", "sobre", "son", "su", "sus", "tan",
    "te", "tiene", "todo", "todos", "tu", "un", "una", "uno", "unos", "y", "ya", "yo",
    "sirve", "cómo", "cuál", "cuáles", "quiero", "tengo",
]


def indexar(corpus):
    documentos = [f"{d['titulo']}. {d['titulo']}. {d['texto']}" for d in corpus]
    vectorizador = TfidfVectorizer(
        lowercase=True, strip_accents="unicode",
        ngram_range=(1, 2), sublinear_tf=True, stop_words=STOPWORDS_ES,
    )
    return vectorizador, vectorizador.fit_transform(documentos)


vectorizador, matriz = indexar(CORPUS)
print(f"Índice: {matriz.shape[0]} documentos x {matriz.shape[1]} términos")

In [ ]:
def recuperar(pregunta, k=3, umbral=0.05):
    """Devuelve los k fragmentos más parecidos que superen el umbral."""
    puntajes = (matriz @ vectorizador.transform([pregunta]).T).toarray().ravel()
    mejores = np.argsort(-puntajes)[:k]
    return [{**CORPUS[i], "puntaje": float(puntajes[i])}
            for i in mejores if puntajes[i] >= umbral]


for f in recuperar(pregunta_interna):
    print(f"{f['puntaje']:.3f}  (Sesión {f['sesion']}) {f['titulo']}")

## 5. Paso 3: generar con el contexto recuperado

Ahora sí, el paso que faltaba. Armamos el prompt igual que en el notebook 06 —contexto numerado
más pregunta— y esta vez **sí llamamos al modelo**.

In [ ]:
PROMPT_SISTEMA = """Eres el asistente del Módulo V del Diplomado de Ciencia de Datos de la FES Acatlán, UNAM.

Reglas que debes respetar siempre:
1. Responde ÚNICAMENTE con la información de los fragmentos de CONTEXTO que recibes.
2. Cita entre corchetes el número del fragmento que respalda cada afirmación, así: [1].
3. Si el contexto no contiene la respuesta, dilo con claridad y no la completes con
   conocimiento propio. Es preferible admitir que no sabes a inventar.
4. Responde en español, en un tono claro y didáctico, en un máximo de seis oraciones.
5. No inventes números, fechas ni nombres que no estén en el contexto."""


def formatear_contexto(fragmentos):
    return "\n\n".join(
        f"[{n}] (Sesión {f['sesion']} — {f['titulo']}) {f['texto']}"
        for n, f in enumerate(fragmentos, start=1)
    )


def preguntar_con_rag(pregunta, k=3, umbral=0.05, prompt_sistema=PROMPT_SISTEMA,
                      temperatura=0.2, mostrar_contexto=False):
    fragmentos = recuperar(pregunta, k=k, umbral=umbral)

    if not fragmentos:
        return ("No encontré nada sobre eso en las notas del módulo, "
                "así que prefiero no responder.")

    contenido = f"CONTEXTO:\n{formatear_contexto(fragmentos)}\n\nPREGUNTA: {pregunta}"
    if mostrar_contexto:
        print(contenido, "\n" + "=" * 70 + "\n")

    return preguntar_al_modelo(
        [{"role": "system", "content": prompt_sistema},
         {"role": "user",   "content": contenido}],
        temperatura=temperatura,
    )


print(preguntar_con_rag(pregunta_interna))

Compara esta respuesta con la de la sección 2, sobre exactamente la misma pregunta:

- Ahora **acierta**, porque el dato está en el contexto.
- Ahora **cita** de dónde salió cada afirmación, así que alguien puede ir a verificarla. Esto es
  la diferencia entre una respuesta útil y una respuesta que hay que creerle.
- Y no hubo que reentrenar nada: cambiamos el contexto, no el modelo.

In [ ]:
# Corre esta celda para ver el prompt completo que recibió el modelo.
_ = preguntar_con_rag("¿Cuál es la diferencia entre Ridge y Lasso?", mostrar_contexto=True)
print(_)

## 6. El prompt de sistema, medido

Aquí está el experimento central de la sesión. Vamos a **dejar fijo el contexto recuperado y
cambiar solo el prompt de sistema**, para ver que las instrucciones no son un adorno: son lo que
decide si el sistema es confiable.

### El experimento

Le vamos a hacer una pregunta **cuya respuesta no está en el corpus**, pero que recupera
fragmentos temáticamente cercanos. Es el caso peligroso que identificamos en el notebook 06: el
recuperador entrega algo, y el modelo tiene que decidir qué hacer con material que no contesta la
pregunta.

In [ ]:
pregunta_trampa = "¿Cuántos alumnos reprobaron el Módulo V el año pasado y cuál fue el promedio del grupo?"

print("Fragmentos que recupera el sistema:")
for f in recuperar(pregunta_trampa, k=3, umbral=0.0):
    print(f"   {f['puntaje']:.3f}  {f['titulo']}")

In [ ]:
PROMPTS = {
    "A · Sin instrucciones":
        "Eres un asistente del Diplomado de Ciencia de Datos. Responde en español.",

    "B · Con contexto, pero permisivo":
        ("Eres un asistente del Diplomado de Ciencia de Datos. Usa el contexto que se te da "
         "para responder la pregunta del usuario. Responde en español."),

    "C · Estricto (el que usa nuestra app)":
        PROMPT_SISTEMA,
}

fragmentos = recuperar(pregunta_trampa, k=3, umbral=0.0)
contenido = f"CONTEXTO:\n{formatear_contexto(fragmentos)}\n\nPREGUNTA: {pregunta_trampa}"

for nombre, prompt in PROMPTS.items():
    print("=" * 70)
    print(nombre)
    print("=" * 70)
    print(preguntar_al_modelo(
        [{"role": "system", "content": prompt},
         {"role": "user",   "content": contenido}],
        temperatura=0.2,
    ))
    print()

**Qué observar.** El contexto recuperado es **idéntico** en los tres casos; lo único que
cambia son las instrucciones. Típicamente verás que:

- **A** ignora el contexto o lo usa a medias, y con frecuencia produce cifras inventadas con tono
  seguro. No le pedimos que se limitara al contexto, así que no lo hace.
- **B** usa el contexto, pero al no tener prohibido completar con conocimiento propio, tiende a
  "rellenar" lo que falta.
- **C** reconoce que el contexto no contiene la respuesta y lo dice. Es el comportamiento que
  queremos, y lo conseguimos con **una regla escrita en español**, no con más cómputo.

**El punto de fondo:** en un sistema RAG, el prompt de sistema es donde se implementa la política
de honestidad. No es "afinar el tono". Y como todo componente de software, hay que **probarlo con
casos adversos** —preguntas cuya respuesta no existe— y no solo con los casos que sí funcionan.

> **Advertencia honesta.** El prompt de sistema *reduce* mucho las alucinaciones, pero no las
> elimina: es una instrucción en lenguaje natural, no una garantía formal. Si corres la celda
> varias veces puedes encontrar respuestas distintas, y ocasionalmente el prompt estricto también
> fallará. Por eso en la app combinamos **dos** defensas: el umbral de similitud (que ni siquiera
> llama al modelo si no recupera nada) y este prompt.

### 6.1 La temperatura

El otro parámetro que conviene entender antes de poner algo en producción. La temperatura controla
qué tan aleatoria es la elección de cada palabra: con 0 el modelo es casi determinista; con
valores altos explora opciones menos probables.

In [ ]:
pregunta_fija = "¿Para qué sirve el coeficiente de silueta?"

for t in [0.0, 0.7, 1.2]:
    print(f"--- temperatura = {t} ---")
    for intento in range(2):
        print(f"  ({intento + 1}) {preguntar_con_rag(pregunta_fija, temperatura=t)[:220]}...")
    print()

Con temperatura 0 las dos corridas salen casi iguales; conforme sube, empiezan a divergir.

**Cuál usar.** Para un asistente que responde sobre documentos —lo que estamos construyendo—
conviene temperatura **baja** (0 a 0.3): queremos que la misma pregunta dé la misma respuesta y
que el modelo se apegue al contexto. Temperaturas altas tienen sentido para generar ideas o
variantes creativas, no para responder qué dice un reglamento.

## 7. Elegir el recuperador con datos (y con la restricción del servidor)

Nos falta una decisión de diseño antes de desplegar: **¿TF-IDF o embeddings?** En el notebook 06
usamos embeddings y aquí usamos TF-IDF sin justificarlo. Vamos a decidirlo midiendo.

Para medir necesitamos un conjunto de preguntas con la respuesta correcta conocida. Lo dividimos
en dos, porque la distinción resultó ser decisiva:

- **Preguntas normales:** usan el vocabulario del corpus (*"¿qué es el coeficiente de silueta?"*).
- **Preguntas parafraseadas:** preguntan lo mismo con otras palabras (*"¿cómo decido cuántos
  grupos usar?"*).

In [ ]:
EVALUACION = [
    # (pregunta, título del fragmento que debería recuperarse)
    ("¿Qué método de regresión sirve para seleccionar variables?", "Regresión Lasso y selección de variables"),
    ("¿Qué hago si mi variable dependiente es una calificación del 1 al 5?", "Logit ordinal"),
    ("¿Para qué sirve el prompt de sistema?", "Prompt de sistema"),
    ("¿Cuánto vale cada tarea en la calificación final?", "Evaluación del módulo"),
    ("¿Por qué la R cuadrada no sirve para comparar modelos?", "Bondad de ajuste y R cuadrada"),
    ("¿Qué mide la similitud coseno?", "Similitud coseno"),
    ("¿Cómo comparo dos series de tiempo desfasadas?", "Deformación dinámica del tiempo (DTW)"),
    ("¿Qué riesgos tiene darle herramientas a un modelo?", "Agentes y uso de herramientas"),
]

PARAFRASEADAS = [
    ("¿Cómo evito que mi modelo memorice en vez de aprender?", "Sobreajuste y separación de la muestra"),
    ("Quiero que el chatbot no invente datos, ¿qué hago?", "Generación aumentada con recuperación (RAG)"),
    ("Tengo muchas columnas y quiero graficarlas en un plano", "Análisis de componentes principales"),
    ("¿Cómo junto clientes parecidos sin tener etiquetas?", "K-medias"),
    ("La palabra banco no significa lo mismo en todas las oraciones", "Embeddings de token frente a embeddings de oración"),
    ("¿Qué técnica castiga los coeficientes grandes hasta volverlos cero?", "Regresión Lasso y selección de variables"),
    ("Mis encuestados respondieron en una escala del uno al cinco", "Logit ordinal"),
    ("¿Cómo le digo al modelo qué papel debe tomar antes de que el usuario escriba?", "Prompt de sistema"),
    ("¿Cómo sé si mi agrupamiento quedó bien?", "Coeficiente de silueta y número de grupos"),
    ("Necesito que responda igual cada vez que le pregunto lo mismo", "Parámetros de un modelo de lenguaje"),
]

FUERA_DE_CORPUS = [
    "¿Cuál es la mejor receta de tacos al pastor?",
    "¿Quién ganó el mundial de 2022?",
    "¿Cómo llego al aeropuerto de la Ciudad de México?",
]

titulos = [d["titulo"] for d in CORPUS]


def evaluar(nombre, puntajes_de):
    filas = []
    for etiqueta, conjunto in [("normales", EVALUACION), ("parafraseadas", PARAFRASEADAS)]:
        aciertos_1 = aciertos_3 = 0
        peor = 1.0
        for pregunta, esperado in conjunto:
            s = puntajes_de(pregunta)
            orden = [titulos[i] for i in np.argsort(-s)]
            aciertos_1 += (orden[0] == esperado)
            aciertos_3 += (esperado in orden[:3])
            peor = min(peor, float(s.max()))
        filas.append({
            "recuperador": nombre, "preguntas": etiqueta,
            "acierto@1": f"{aciertos_1}/{len(conjunto)}",
            "acierto@3": f"{aciertos_3}/{len(conjunto)}",
            "peor puntaje": round(peor, 3),
        })
    mejor_fuera = max(float(puntajes_de(q).max()) for q in FUERA_DE_CORPUS)
    for fila in filas:
        fila["mejor fuera de corpus"] = round(mejor_fuera, 3)
    return filas


def puntajes_tfidf(pregunta):
    return (matriz @ vectorizador.transform([pregunta]).T).toarray().ravel()


resultados = evaluar("TF-IDF", puntajes_tfidf)
pd.DataFrame(resultados)

### 7.1 ¿Y si usáramos embeddings?

La celda siguiente repite exactamente la misma evaluación con un modelo de embeddings
multilingüe, el mismo tipo que usamos en el bloque anterior.

> **Descarga de ~470 MB.** En Colab tarda un par de minutos. Si el tiempo de la sesión aprieta,
> puedes saltarte esta celda: los resultados que obtuvimos están en la tabla de más abajo.

In [ ]:
EJECUTAR_EMBEDDINGS = True     # ponlo en False para saltarte la descarga

if EJECUTAR_EMBEDDINGS:
    import os
    os.environ["USE_TF"] = "0"          # antes de importar, como en el bloque de embeddings
    asegurar("sentence-transformers", "sentence_transformers")
    from sentence_transformers import SentenceTransformer

    modelo_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    documentos = [f"{d['titulo']}. {d['titulo']}. {d['texto']}" for d in CORPUS]
    E = modelo_emb.encode(documentos, normalize_embeddings=True)

    def puntajes_embeddings(pregunta):
        return E @ modelo_emb.encode(pregunta, normalize_embeddings=True)

    resultados += evaluar("Embeddings", puntajes_embeddings)

pd.DataFrame(resultados)

### 7.2 La decisión

Estos son los resultados que obtuvimos al preparar la sesión (los tuyos deberían coincidir):

| Recuperador | Preguntas normales | Parafraseadas (@1) | Parafraseadas (@3) |
|---|---|---|---|
| TF-IDF | **8/8** | 6/10 | **8/10** |
| Embeddings multilingües | 7/8 | 6/10 | 7/10 |

**El resultado es el contrario de lo que dicta la intuición: TF-IDF no pierde.** Empata en las
preguntas parafraseadas y gana por poco en las demás, siendo diez veces más ligero. Antes de
generalizar, entendamos por qué pasa esto aquí:

1. **El corpus es chico** (29 fragmentos). Con tan pocos documentos hay poca oportunidad de que
   dos compitan por las mismas palabras, así que coincidir literalmente basta para desempatar.
2. **Los fragmentos son ricos en vocabulario.** Cada uno repite los términos clave de su tema y
   menciona sus sinónimos, lo que le da mucho material a TF-IDF para engancharse. Escribir bien el
   corpus mejoró la recuperación más que cambiar de algoritmo.
3. **Nuestras preguntas son de estudiantes del curso**, que usan las palabras del curso.

> **Cuidado con sobreinterpretar.** Son 18 preguntas: las diferencias de uno o dos aciertos están
> dentro del ruido y no permiten declarar un ganador. Lo que sí sostiene la evidencia es lo
> importante: **no hay una brecha que justifique el costo del modelo pesado en este caso.**

Con un corpus de miles de documentos, con textos escuetos, o con usuarios que preguntan con
vocabulario ajeno al de los documentos, la balanza se inclina hacia los embeddings. La forma de
saberlo es la que acabamos de usar: **armar un conjunto de preguntas con respuesta conocida y
medir**, en vez de elegir por reputación de la técnica.

### La restricción que cierra la discusión

Aun si los embeddings hubieran ganado, había un límite duro. Medimos la memoria que consume cada
opción:

| Recuperador | RAM medida | ¿Cabe en Streamlit Community Cloud (1 GB)? |
|---|---|---|
| TF-IDF | ~150 MB | Sí, con holgura |
| `all-MiniLM-L6-v2` (solo inglés) | 597 MB | Apenas, y obligaría a traducir el corpus |
| `paraphrase-multilingual-MiniLM-L12-v2` | 1,372 MB | **No** |

El plan gratuito da **1 GB de RAM**. El modelo multilingüe no cabe y nuestro corpus está en
español. La app usa TF-IDF, y ahora sabemos que además no estamos sacrificando calidad.

**La lección de ingeniería de la sesión:** la técnica más nueva no es automáticamente la mejor
para tu problema, y el lugar donde vas a desplegar es parte del diseño, no un detalle del final.
Medir cuesta veinte minutos y evita meses de complejidad innecesaria.

## 8. Del notebook a la aplicación

Ya tienes las tres piezas funcionando. Lo que sigue, en la segunda mitad de la sesión, es
empaquetarlas en una aplicación web y **desplegarla en internet con una URL tuya**.

El código de la app es esencialmente el de este notebook, reorganizado en tres archivos:

| Archivo | Contiene |
|---|---|
| `corpus_diplomado.py` | El corpus de la sección 3, sin cambios. |
| `rag_core.py` | `indexar`, `recuperar`, el prompt de sistema y la llamada al modelo: las secciones 4, 5 y 6. |
| `streamlit_app.py` | Solo la interfaz: barra lateral, chat, expansores. |

Separar la lógica de la interfaz no es un capricho: permite probar `rag_core.py` sin levantar la
aplicación, que es como se prueba el software de verdad.

**Continúa en `09 App-RAG-Streamlit/README.md`.**

## Para pensar

1. En la sección 2 el modelo respondió sin contexto y en la 5 con contexto, a la misma pregunta.
   Además de acertar, la segunda respuesta trae citas. Explica por qué la posibilidad de **citar
   la fuente** cambia la naturaleza del sistema, y por qué eso importa especialmente en un trabajo
   académico o de política pública.

2. El experimento de la sección 6 mostró que un prompt de sistema estricto evita que el modelo
   invente. Un compañero concluye: "entonces con un buen prompt ya no hay alucinaciones". Escribe
   una respuesta de tres o cuatro líneas explicando por qué esa conclusión es demasiado optimista,
   apoyándote en lo que dice la advertencia de esa sección.

3. En la sección 7 medimos que TF-IDF no pierde contra los embeddings en este corpus, pese a que
   los embeddings son la técnica más sofisticada. Da **dos** características que tendría que tener
   un corpus distinto para que el resultado se invirtiera, y explica por qué en ese caso la
   coincidencia literal de palabras dejaría de ser suficiente.

4. La app rechaza la pregunta cuando ningún fragmento supera el umbral. Al calibrarlo encontramos
   que la peor pregunta válida puntúa 0.113 y la mejor pregunta fuera de corpus también 0.113: **no
   hay un umbral que separe perfectamente**. Dado eso, ¿preferirías un umbral alto o bajo para un
   asistente que responde dudas de alumnos? ¿Y para uno que responde consultas médicas? Explica qué
   cambia entre los dos casos.

5. **Ejercicio.** Agrega tres fragmentos nuevos al `CORPUS` con información que el modelo no pueda
   saber (las fechas de entrega de tus tareas, el nombre de tu equipo de proyecto, tu tema de
   investigación). Vuelve a indexar y pregunta por ellos. ¿Los recupera? Si no, revisa si el título
   que les pusiste contiene las palabras con las que preguntaste.

6. **Ejercicio.** Modifica `PROMPT_SISTEMA` para que el asistente responda **siempre en dos
   oraciones y termine con una pregunta que invite a profundizar**. Verifica que sigue cumpliendo
   las reglas 1 a 3 (no inventar, citar, admitir cuando no sabe): agregar instrucciones puede
   debilitar las que ya estaban.